# Sensitivity Analysis Tutorial

In this notebook, we perform a sensitivity analysis on **HVAC electricity demand**
to understand which input features have the strongest influence on energy consumption.

The analysis focuses on answering three simple questions:

1. Which features are most strongly related to HVAC energy demand?
2. Which features does a machine learning model rely on the most?
3. Do different analysis methods lead to consistent conclusions?

To answer these questions, we use several commonly adopted techniques:
- **Pearson correlation** for linear relationships
- **Permutation importance** with a Random Forest model
- **SHAP values** for model interpretability

This notebook is designed as a **step-by-step tutorial**, where each section explains
both *what is being done* and *why it is useful*.


## Step 1 — Import Libraries and Setup

In this notebook, we perform sensitivity analysis on HVAC energy demand.
Before starting the analysis, we first import all required Python libraries.

- **pandas / numpy** are used for data handling and numerical operations.
- **matplotlib / seaborn** are used for visualization.
- **RandomForestRegressor** is used as a non-linear model for feature importance analysis.
- **permutation_importance** helps evaluate how important each feature is to the model.
- **shap** is used for model explainability and understanding feature impact.

Finally, we set a consistent plotting style to make all figures easier to read.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

import shap

sns.set(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 6)

## Step 2 — Load Dataset and Sampling

In this step, we load the original HVAC dataset from a CSV file.

The full dataset is relatively large, so for efficiency and faster experimentation,
we randomly sample **20,000 data points** from the original data.
A fixed random seed is used to ensure reproducibility.

This sampling strategy significantly reduces computation time
while still preserving the overall data distribution.


In [ ]:
df = pd.read_csv("chiller_data.csv")   
df = df.sample(n=20000, random_state=42).reset_index(drop=True)
print("Shape:", df.shape)
df.head()

## Step 3 — Define Target Variable and Features

In this step, we define the **prediction target** and select the input features.

- The target variable is **HVAC_electricity_demand_rate**, which represents
  the real-time energy consumption of the HVAC system.
- Some columns are explicitly removed from the feature set:
  - Variables that are directly related to total HVAC electricity
  - Time indices such as month, day, and hour, which are not used as inputs here

After dropping these columns, the remaining variables are used as input features
for sensitivity analysis.


In [ ]:
target_col = "HVAC_electricity_demand_rate"

drop_cols = [
    target_col,
    "total_electricity_HVAC",
    "month",
    "day_of_month",
    "hour",
]

drop_cols = [c for c in drop_cols if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols]

print("Target:", target_col)
print("Features:", feature_cols)

X = df[feature_cols]
y = df[target_col]

## Step 4 — Pearson Correlation Analysis

In this step, we use **Pearson correlation** to measure the linear relationship
between each input feature and the HVAC electricity demand.

For easier comparison, we take the **absolute value** of the correlation
and sort the features from most to least correlated.
This allows us to quickly identify which variables are most strongly related
to HVAC energy consumption in a linear sense.

The bar plot below provides a clear visual ranking of feature importance
based on Pearson correlation.


In [ ]:
corr_series = df[feature_cols + [target_col]].corr()[target_col].drop(target_col)
corr_df = corr_series.abs().sort_values(ascending=False).reset_index()
corr_df.columns = ["feature", "abs_pearson_corr"]

display(corr_df)

plt.figure(figsize=(8, len(corr_df)*0.45))
sns.barplot(data=corr_df, x="abs_pearson_corr", y="feature")
plt.title("Pearson Correlation with HVAC Energy Demand")
plt.xlabel("Absolute Pearson Correlation")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Step 5 — Correlation Heatmap

After examining the correlation between each feature and the target,
we further visualize the **pairwise correlations** using a heatmap.

This plot shows the correlation between:
- input features themselves, and
- each feature and the HVAC electricity demand.

The heatmap helps identify:
- strongly correlated feature groups,
- potential redundancy between variables,
- and whether multiple features capture similar information.

Color intensity indicates correlation strength,
with red representing positive correlation and blue representing negative correlation.


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[feature_cols + [target_col]].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## Step 6 — Train a Random Forest Model

In this step, we train a **Random Forest regression model** using the selected features.
The dataset is split into training and testing sets, with 80% used for training
and 20% reserved for evaluation.

Random Forest is chosen because it can capture **non-linear relationships**
and interactions between features, making it suitable for sensitivity analysis.

We report the **R² score** on both the training and test sets to ensure
that the model generalizes reasonably well and is not severely overfitting.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Train R^2:", rf.score(X_train, y_train))
print("Test  R^2:", rf.score(X_test, y_test))


## Step 7 — Permutation Importance

In this step, we evaluate feature importance using **permutation importance**
based on the trained Random Forest model.

The idea is simple:
- We randomly shuffle one feature at a time in the test set.
- If the model performance (R²) drops significantly, that feature is important.

Here, importance is measured as the **mean decrease in R²** across multiple shuffles.
A larger decrease indicates a stronger influence on the model prediction.

The bar plot below shows the permutation importance results,
sorted from least to most important for better visualization.


In [ ]:
# --- Permutation Importance (improved version) ---

perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=8,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
})

# Sort from small → large for better visualization
perm_df_sorted = perm_df.sort_values("importance_mean", ascending=True)

display(perm_df_sorted)

# Improved plot
plt.figure(figsize=(10, 6))
sns.barplot(
    data=perm_df_sorted,
    x="importance_mean",
    y="feature",
    palette="Blues_r"
)

plt.title("Permutation Importance (Random Forest)", fontsize=16)
plt.xlabel("Mean Decrease in R²", fontsize=14)
plt.ylabel("Feature", fontsize=14)

# Add numeric labels to bars
for i, v in enumerate(perm_df_sorted["importance_mean"]):
    plt.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()


## Step 8 — SHAP Analysis

In this step, we use **SHAP (SHapley Additive exPlanations)** to interpret
how each feature influences the model prediction.

SHAP explains the model output by showing:
- whether a feature pushes the prediction **higher or lower**, and
- how strong that influence is across different samples.

We use a TreeExplainer, which is well-suited for tree-based models
such as Random Forest.

Two types of plots are shown:
- **Summary scatter plot**: shows both the direction and magnitude of feature impact.
- **Summary bar plot**: shows the average importance of each feature,
  which is especially useful for reporting.


In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Summary scatter plot
shap.summary_plot(shap_values, X_test)

# Summary bar plot
shap.summary_plot(shap_values, X_test, plot_type="bar")

## Step 9 — Summary and Comparison of Results

In this final step, we summarize the sensitivity analysis results
by comparing **Pearson correlation** and **permutation importance** side by side.

- Pearson correlation reflects the **linear relationship** between each feature
  and HVAC electricity demand.
- Permutation importance reflects how much the **trained model relies on each feature**
  when making predictions.

By combining these two perspectives into a single table,
we can easily check whether different methods lead to consistent conclusions.

Features that rank highly in both metrics can be considered
the most influential drivers of HVAC energy demand.


In [ ]:
pearson_s = corr_df.set_index("feature")["abs_pearson_corr"]
perm_s = perm_df.set_index("feature")["importance_mean"]

summary = pd.concat([pearson_s, perm_s], axis=1)
summary.columns = ["pearson_corr", "perm_importance"]
summary = summary.fillna(0).sort_values("perm_importance", ascending=False)

display(summary)

# Noise Robustness Analysis Tutorial

In this notebook, we evaluate the **robustness of an LSTM-based HVAC energy prediction model**
under different levels of input noise.

Instead of retraining the model multiple times, we:
- train the LSTM **once** on clean data, and
- test the same trained model under increasing levels of **Gaussian noise**.

This experiment helps answer a practical question:

> How sensitive is the trained LSTM model to noisy or imperfect input signals?

Model robustness is evaluated using **R², RMSE, and MAE** on the test set,
with a focus on identifying when model performance begins to break down.


## Step 1 — Noise Robustness Experiment Setup

In this section, we set up the environment and experimental configuration for the noise robustness analysis.

The goal of this experiment is simple:
- Train the LSTM model **once** using clean data
- Add Gaussian noise **only at test time**
- Observe how prediction performance degrades as noise increases

Below, we define:
- File paths and output folders
- Model and training hyperparameters
- Noise levels for the robustness sweep
- Random seeds for reproducibility


In [ ]:
mport os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# =====================================================
# Configuration
# =====================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_PATH = os.path.join(BASE_DIR, "chiller_data.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_noise_sweep_fast")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "HVAC_electricity_demand_rate"
LOOKBACK = 24
BATCH_SIZE = 64      
EPOCHS = 10          
SEED = 42

# ✅ Train subsample for speed (keeps time order by sorting index after sampling)
TRAIN_SAMPLE_SIZE = 50000   

# Noise sweep (test only)
NOISE_LEVELS = np.arange(0.0, 3.01, 0.1)

# Optional: early stop training if it’s clearly converged
EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 1e-4


# =====================================================
# Reproducibility
# =====================================================
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)


## Step 2 — Model Architecture and Helper Functions

In this step, we define the core components used in the noise robustness experiment.

This includes:
- The LSTM model architecture for HVAC energy forecasting
- A utility function to construct time-series sequences with a fixed lookback window
- A simple early stopping mechanism to speed up training

At this stage, no data is loaded and no training is performed.
We only prepare the building blocks that will be used in later steps.


In [ ]:

# =====================================================
# Model
# =====================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, 64, batch_first=True)
        self.lstm2 = nn.LSTM(64, 32, batch_first=True)
        self.fc1 = nn.Linear(32, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x = x[:, -1, :]
        x = self.relu(self.fc1(x))
        return self.fc2(x)


# =====================================================
# Utilities
# =====================================================
def make_sequences(x_concat, y, lookback):
    """
    x_concat: (N, F) where F includes [scaled_X + scaled_y_lag]
    y: (N, 1) scaled target
    """
    xs, ys = [], []
    for t in range(lookback, len(x_concat)):
        xs.append(x_concat[t - lookback:t])
        ys.append(y[t])
    return np.array(xs), np.array(ys)


class SimpleEarlyStop:
    def __init__(self, patience=2, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.count = 0

    def step(self, metric):
        if self.best is None or metric < self.best - self.min_delta:
            self.best = metric
            self.count = 0
            return False
        self.count += 1
        return self.count >= self.patience

## Step 3 — Data Loading and Preprocessing

In this step, we load the chiller dataset and prepare it for LSTM training.

The preprocessing pipeline includes:
- Loading the raw CSV file
- Splitting the data into training and test sets based on time order
- Subsampling the training data to speed up experiments
- Scaling input features and target values
- Constructing rolling window sequences for the LSTM model

All preprocessing steps are applied consistently to ensure a fair robustness evaluation later.


In [ ]:
# =====================================================
# Load & preprocess data
# =====================================================
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Cannot find CSV at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c != TARGET_COL]

# Time-order split
split = int(0.8 * len(df))
df_train_full = df.iloc[:split].copy()
df_test = df.iloc[split:].copy()

# ✅ Subsample train for speed (keep temporal variety, then sort back by index)
if TRAIN_SAMPLE_SIZE is not None and TRAIN_SAMPLE_SIZE < len(df_train_full):
    df_train = (
        df_train_full
        .sample(n=TRAIN_SAMPLE_SIZE, random_state=SEED)
        .sort_index()
        .copy()
    )
else:
    df_train = df_train_full

print(f"Device: {device}")
print(f"Train size: {len(df_train)} (from {len(df_train_full)}), Test size: {len(df_test)}")
print(f"Features: {len(feature_cols)}, Lookback: {LOOKBACK}")
print("Fitting scalers on TRAIN (subsampled) ...")

scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_train_raw = df_train[feature_cols].values
y_train_raw = np.log1p(df_train[TARGET_COL].values).reshape(-1, 1)

scaler_x.fit(x_train_raw)
scaler_y.fit(y_train_raw)

x_train_scaled = scaler_x.transform(x_train_raw)
y_train_scaled = scaler_y.transform(y_train_raw)

# Build training sequences (concat target lag into input)
x_train_concat = np.concatenate([x_train_scaled, y_train_scaled], axis=1)
x_train_seq, y_train_seq = make_sequences(x_train_concat, y_train_scaled, LOOKBACK)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(x_train_seq), torch.FloatTensor(y_train_seq)),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Train sequences: {x_train_seq.shape}")

## Step 4 — Train Baseline LSTM (Noise = 0)

In this step, we train the LSTM model using clean data without any added noise.

This trained model serves as the baseline for the robustness experiment:
- The model is trained **only once**
- No noise is added during training
- The same trained model will be reused for all noise levels during testing

By fixing the trained model, we can isolate the effect of input noise on prediction performance.


In [ ]:
# =====================================================
# Train once (noise = 0)
# =====================================================
model = LSTMModel(input_size=x_train_seq.shape[2]).to(device)
optimizer = optim.Adam(model.parameters())
criterion = nn.L1Loss()

early_stop = SimpleEarlyStop(patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA)

train_losses = []
print("\nTraining baseline LSTM (noise = 0)...")

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    avg_loss = float(np.mean(batch_losses))
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train MAE (scaled log space): {avg_loss:.4f}")

    if early_stop.step(avg_loss):
        print("Early stop: training loss plateaued.")
        break

# Save training loss curve
plt.figure(figsize=(7, 4))
plt.plot(train_losses, marker="o")
plt.title("Training Loss (baseline, noise=0)")
plt.xlabel("Epoch")
plt.ylabel("MAE (scaled log space)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "train_loss_curve.png"))
plt.close()

print("Training done.\n")


## Step 5 — Noise Sensitivity Sweep (Test-Time Only)

In this step, we evaluate the robustness of the trained LSTM model under input noise.

The key idea is:
- The model is **not retrained**
- Gaussian noise is added **only to test inputs**
- Prediction performance is measured for different noise levels

For each noise standard deviation, we compute:
- R² score
- RMSE
- MAE

This allows us to observe how prediction quality degrades as input noise increases.


In [ ]:
# =====================================================
# Noise sweep (TEST only)
# =====================================================
model.eval()

# Prepare clean test scaled arrays
x_test_raw = df_test[feature_cols].values
y_test_raw = np.log1p(df_test[TARGET_COL].values).reshape(-1, 1)

x_test_scaled_clean = scaler_x.transform(x_test_raw)
y_test_scaled = scaler_y.transform(y_test_raw)

results = []
print("Running noise sensitivity (test-time only)...")

for i, noise_std in enumerate(NOISE_LEVELS):
    # Make noise reproducible per noise level
    np.random.seed(SEED + i)

    noise = np.random.normal(loc=0.0, scale=float(noise_std), size=x_test_scaled_clean.shape)
    x_test_noisy = x_test_scaled_clean + noise

    # Build sequences (still using true y_lag scaled as in your original design)
    x_test_concat = np.concatenate([x_test_noisy, y_test_scaled], axis=1)
    x_seq, y_seq = make_sequences(x_test_concat, y_test_scaled, LOOKBACK)

    with torch.no_grad():
        preds_scaled = model(torch.FloatTensor(x_seq).to(device)).cpu().numpy()

    # Inverse transform to original scale
    preds_log = scaler_y.inverse_transform(preds_scaled)
    preds = np.expm1(preds_log).ravel()

    y_true_log = scaler_y.inverse_transform(y_seq)
    y_true = np.expm1(y_true_log).ravel()

    r2 = float(r2_score(y_true, preds))
    rmse = float(np.sqrt(mean_squared_error(y_true, preds)))
    mae = float(mean_absolute_error(y_true, preds))

    collapsed = (r2 < 0.0)

    results.append({
        "noise_std": float(noise_std),
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "collapsed_R2<0": collapsed
    })

    print(f"noise={noise_std:.2f}  R2={r2:.3f}  RMSE={rmse:.2f}  MAE={mae:.2f}  collapsed={collapsed}")

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "noise_sensitivity_results.csv"), index=False)

## Step 6 — Results Visualization and Model Robustness

In this step, we visualize the results of the noise sensitivity experiment.

We focus on three aspects:
- The relationship between noise level and test R²
- The point where model performance collapses (R² < 0)
- Example forecast comparisons under different noise levels

These visualizations help us understand how robust the LSTM model is to noisy inputs.


In [ ]:
# Plot noise vs R2
plt.figure(figsize=(8, 5))
plt.plot(results_df["noise_std"], results_df["R2"], marker="o")
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.xlabel("Gaussian noise std (added to scaled X at test time)")
plt.ylabel("Test R² (original scale)")
plt.title("LSTM Robustness: Noise vs R² (train once, test noisy)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "noise_vs_r2.png"))
plt.close()

# Optional: save example forecast plot at a few noise levels
example_levels = [0.0, 0.5, 1.0, 2.0]
for noise_std in example_levels:
    # find closest
    idx = int(np.argmin(np.abs(results_df["noise_std"].values - noise_std)))
    noise_std = float(results_df.loc[idx, "noise_std"])

    np.random.seed(SEED + 1000 + idx)
    noise = np.random.normal(loc=0.0, scale=noise_std, size=x_test_scaled_clean.shape)
    x_test_noisy = x_test_scaled_clean + noise
    x_test_concat = np.concatenate([x_test_noisy, y_test_scaled], axis=1)
    x_seq, y_seq = make_sequences(x_test_concat, y_test_scaled, LOOKBACK)

    with torch.no_grad():
        preds_scaled = model(torch.FloatTensor(x_seq).to(device)).cpu().numpy()

    preds = np.expm1(scaler_y.inverse_transform(preds_scaled)).ravel()
    y_true = np.expm1(scaler_y.inverse_transform(y_seq)).ravel()

    plt.figure(figsize=(12, 4))
    n = 500
    plt.plot(y_true[:n], label="Actual")
    plt.plot(preds[:n], label=f"Pred (noise={noise_std:.2f})", alpha=0.8)
    plt.title(f"Forecast Comparison (first {n} points) - noise={noise_std:.2f}")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"forecast_noise_{noise_std:.2f}.png"))
    plt.close()

# Print breaking point
collapsed_rows = results_df[results_df["R2"] < 0.0]
if len(collapsed_rows) > 0:
    first = collapsed_rows.iloc[0]
    print(f"\nFirst collapse (R² < 0) at noise_std ≈ {first['noise_std']:.2f}")
else:
    print("\nNo collapse observed (R² never dropped below 0).")

print("\nDone. Outputs saved to:", OUTPUT_DIR)
